# CreditLens — 02 · Feature Engineering

Turn the three raw tables into **one model-ready frame** (one row per `SK_ID_CURR`, no leakage).
The logic lives in `creditlens/data/features.py`; this notebook demonstrates and **sanity-checks** it.

Steps: application ratios → aggregate `bureau` → aggregate `previous_application` → join → checks.

## 0 · Setup

In [1]:
import sys; sys.path.insert(0, '..')

import pandas as pd

from creditlens.data.load import load_application, load_bureau, load_previous_application
from creditlens.data.features import (
    add_application_features, aggregate_bureau, aggregate_previous, build_features,
)
from creditlens.config import TARGET, ID_COL

pd.set_option('display.max_columns', 60)

app = load_application()
bureau = load_bureau()
prev = load_previous_application()
print('app', app.shape, '| bureau', bureau.shape, '| prev', prev.shape)

app (307511, 122) | bureau (1716428, 17) | prev (1670214, 37)


## 1 · Application features
Affordability ratios + decoded age/employment. Divide-by-zero → inf is cleaned to NaN.
We check the new columns exist and peek at their values.

In [2]:
app_fe = add_application_features(app)
new_cols = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
            'GOODS_CREDIT_RATIO', 'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_TO_AGE']
print('added:', new_cols)
print('any inf left?', app_fe[new_cols].isin([float('inf'), float('-inf')]).any().any())
app_fe[new_cols].describe().round(3)

added: ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM', 'GOODS_CREDIT_RATIO', 'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_TO_AGE']
any inf left? False


,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_TERM,GOODS_CREDIT_RATIO,AGE_YEARS,EMPLOYED_YEARS,EMPLOYED_TO_AGE
count,307511.000,307499.000,307499.000,307233.000,307511.000,252137.000,252137.000
mean,3.958,0.181,0.054,0.901,43.937,6.532,0.157
std,2.690,0.095,0.022,0.097,11.956,6.406,0.134
min,0.005,0.000,0.022,0.167,20.518,-0.000,-0.000
25%,2.019,0.115,0.037,0.835,34.008,2.101,0.056
50%,3.265,0.163,0.050,0.894,43.151,4.515,0.119
75%,5.160,0.229,0.064,1.000,53.923,8.699,0.219
max,84.737,1.876,0.124,6.667,69.121,49.074,0.729


## 2 · Aggregate `bureau` → one row per applicant
Counts, active loans, credit sums, overdue stats. `BUREAU_` prefix keeps them traceable.

In [3]:
bureau_agg = aggregate_bureau(bureau)
print('bureau_agg:', bureau_agg.shape, '| unique ids:', bureau_agg[ID_COL].nunique())
assert bureau_agg[ID_COL].is_unique, 'aggregation must yield one row per applicant'
bureau_agg.head(3)

bureau_agg: (305811, 11) | unique ids: 305811


,SK_ID_CURR,BUREAU_COUNT,BUREAU_ACTIVE_COUNT,BUREAU_AMT_CREDIT_SUM_sum,BUREAU_AMT_CREDIT_SUM_mean,BUREAU_AMT_CREDIT_SUM_DEBT_sum,BUREAU_DAY_OVERDUE_sum,BUREAU_DAY_OVERDUE_max,BUREAU_DAYS_CREDIT_mean,BUREAU_DAYS_CREDIT_min,BUREAU_ACTIVE_RATE
0,100001,7,3,1453365.000,207623.571429,596686.5,0,0,-735.00,-1572,0.428571
1,100002,8,2,865055.565,108131.945625,245781.0,0,0,-874.00,-1437,0.250000
2,100003,4,1,1017400.500,254350.125000,0.0,0,0,-1400.75,-2586,0.250000


## 3 · Aggregate `previous_application` → one row per applicant
Count, approval rate, application/credit amounts, mean term. `PREV_` prefix.

In [4]:
prev_agg = aggregate_previous(prev)
print('prev_agg:', prev_agg.shape, '| unique ids:', prev_agg[ID_COL].nunique())
assert prev_agg[ID_COL].is_unique
prev_agg.head(3)

prev_agg: (338857, 8) | unique ids: 338857


,SK_ID_CURR,PREV_COUNT,PREV_APPROVED_RATE,PREV_AMT_APPLICATION_mean,PREV_AMT_APPLICATION_max,PREV_AMT_CREDIT_mean,PREV_AMT_CREDIT_max,PREV_CNT_PAYMENT_mean
0,100001,1,1.0,24835.5,24835.5,23787.0,23787.0,8.0
1,100002,1,1.0,179055.0,179055.0,179055.0,179055.0,24.0
2,100003,3,1.0,435436.5,900000.0,484191.0,1035882.0,10.0


## 4 · Build the full feature frame
`build_features` runs all three steps and left-joins. Count features for no-history applicants are
filled with 0 (truly zero, not unknown); amount/mean features stay NaN for the imputer.

In [5]:
df = build_features(app, bureau, prev)
print('feature frame:', df.shape)
print('new BUREAU_/PREV_ columns:', [c for c in df.columns if c.startswith(('BUREAU_', 'PREV_'))])

feature frame: (307511, 146)
new BUREAU_/PREV_ columns: ['BUREAU_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_AMT_CREDIT_SUM_sum', 'BUREAU_AMT_CREDIT_SUM_mean', 'BUREAU_AMT_CREDIT_SUM_DEBT_sum', 'BUREAU_DAY_OVERDUE_sum', 'BUREAU_DAY_OVERDUE_max', 'BUREAU_DAYS_CREDIT_mean', 'BUREAU_DAYS_CREDIT_min', 'BUREAU_ACTIVE_RATE', 'PREV_COUNT', 'PREV_APPROVED_RATE', 'PREV_AMT_APPLICATION_mean', 'PREV_AMT_APPLICATION_max', 'PREV_AMT_CREDIT_mean', 'PREV_AMT_CREDIT_max', 'PREV_CNT_PAYMENT_mean']


## 5 · Sanity / leakage checks
The checks that catch silent mistakes before they reach the model:
- row count unchanged (left join didn't duplicate)
- `SK_ID_CURR` still unique
- `TARGET` present, still binary, distribution unchanged
- no engineered feature is perfectly correlated with `TARGET` (would signal leakage)

In [6]:
assert df.shape[0] == app.shape[0], 'row count changed -> join duplicated rows'
assert df[ID_COL].is_unique, 'SK_ID_CURR not unique'
assert set(df[TARGET].unique()) <= {0, 1}, 'TARGET no longer binary'
assert df[TARGET].mean() == app[TARGET].mean(), 'TARGET distribution changed'

# leakage smell test: correlation of new numeric features with TARGET
new = [c for c in df.columns if c.startswith(('BUREAU_', 'PREV_'))
       or c in ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM',
                'GOODS_CREDIT_RATIO', 'AGE_YEARS', 'EMPLOYED_YEARS', 'EMPLOYED_TO_AGE']]
corr = df[new + [TARGET]].corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
assert corr.max() < 0.95, f'suspiciously high corr with TARGET: {corr.idxmax()}={corr.max():.3f}'
print('all checks passed. row count:', df.shape[0])
print('\ntop |corr| with TARGET (sanity — all modest, no leakage):')
print(corr.head(8).round(3))

all checks passed. row count: 307511

top |corr| with TARGET (sanity — all modest, no leakage):
BUREAU_DAYS_CREDIT_mean    0.090
AGE_YEARS                  0.078
BUREAU_ACTIVE_RATE         0.077
BUREAU_DAYS_CREDIT_min     0.075
EMPLOYED_YEARS             0.075
EMPLOYED_TO_AGE            0.068
GOODS_CREDIT_RATIO         0.065
PREV_APPROVED_RATE         0.064
Name: TARGET, dtype: float64


## 6 · Missingness of the new columns
How many applicants lack bureau / previous history (→ NaN in mean/amount features).

In [7]:
newmiss = df[[c for c in df.columns if c.startswith(('BUREAU_', 'PREV_'))]].isna().mean()
print((newmiss * 100).round(1).sort_values(ascending=False).to_string())

BUREAU_AMT_CREDIT_SUM_sum         14.3
BUREAU_DAYS_CREDIT_mean           14.3
BUREAU_AMT_CREDIT_SUM_mean        14.3
BUREAU_AMT_CREDIT_SUM_DEBT_sum    14.3
BUREAU_DAY_OVERDUE_sum            14.3
BUREAU_DAYS_CREDIT_min            14.3
BUREAU_DAY_OVERDUE_max            14.3
BUREAU_ACTIVE_RATE                14.3
PREV_CNT_PAYMENT_mean              5.5
PREV_AMT_CREDIT_max                5.4
PREV_AMT_CREDIT_mean               5.4
PREV_AMT_APPLICATION_mean          5.4
PREV_APPROVED_RATE                 5.4
PREV_AMT_APPLICATION_max           5.4
BUREAU_COUNT                       0.0
BUREAU_ACTIVE_COUNT                0.0
PREV_COUNT                         0.0


## Next
`build_features` is the single entry point Phase 3 (`creditlens/models`) will call to get `X, y`.
Preprocessing (impute/scale/encode) lives **inside the model Pipeline**, not here — so it is fit only
on training folds and never leaks.

# Notebook summary & key insights

## Task
Feature engineering: turn the three raw Home Credit tables into model-ready frames — one row per `SK_ID_CURR`, no leakage. Logic lives in `creditlens/data/features.py`; this notebook demonstrates and checks it.

## Data & Checks
- Inputs: `application_train` (307,511 rows) + `bureau` (1.72M) + `previous_application` (1.67M).
- Verified each aggregate yields one row per applicant (`SK_ID_CURR` unique), the join keeps row count, and `TARGET` is untouched.

## Feature Engineering
- **Application:** affordability ratios (credit/income, annuity/income, credit-term, goods/credit) + decoded `AGE_YEARS`, `EMPLOYED_YEARS`, `EMPLOYED_TO_AGE`. Divide-by-zero `inf` → NaN.
- **bureau → one row:** counts, active count, credit sums, overdue stats (`BUREAU_` prefix).
- **previous_application → one row:** count, approval rate, amounts, mean term (`PREV_` prefix).
- Two outputs: `build_features` (rich 146-col frame for EDA/exploration) and `build_model_matrix` (the **15-feature contract** the served models + frontend use).

## Checks (leakage / correctness)
- Row count unchanged after left-join (no duplication); `SK_ID_CURR` unique; `TARGET` binary + distribution unchanged.
- **Leakage smell test:** max |corr| of any engineered feature with `TARGET` = 0.09 — modest, no leakage.
- No-history applicants: count features filled 0 (truly zero); amount/rate features left NaN for the pipeline imputer (~14% for bureau).

## Insights & Recommendations
- **Insight:** Aggregate *within each applicant's own history* only — never across applicants or using `TARGET` — to stay leakage-free.
- **Insight:** On real data `credit_to_income` correlates ≈ 0 with default (the design's toy model assumed +0.55); trust the data, not the placeholder betas.
- **Recommendation:** Keep preprocessing (impute/scale/encode) in the model `Pipeline`, not in feature building, so it is fit per-fold.
- **Recommendation:** `build_model_matrix` is the single source of truth for the served feature set — the FastAPI request schema must mirror its 15 columns exactly.

## Next
Notebook **03 · Modeling** — wrap each of the 6 models in a leakage-safe `Pipeline` over the 15-feature contract, train with `StratifiedKFold` OOF predictions, compare AUC/KS.